# Sionna 0.19 – Main Ray Tracing Simulation
**Kernel:** `sionna019` · Python 3.10 · Sionna 0.19.2 · TensorFlow 2.15

Migrated from Untitled(1).ipynb (Sionna 2.0) to Sionna 0.19.2 API.
Uses `scene.compute_paths()` and `scene.coverage_map()` (not PathSolver/RadioMapSolver).
GPS / UTM transforms, DEM lookup, TX/RX CSV loading all preserved from original.

## CELL 0 · Environment Setup & Imports

In [ ]:
import os, sys

import sys, json, csv, time, warnings, glob, re
import xml.etree.ElementTree as ET
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import pandas as pd
import matplotlib; matplotlib.rcParams.update({'font.size': 11, 'figure.dpi': 100})
import matplotlib.pyplot as plt
from scipy.spatial import KDTree
from scipy import stats
from scipy.stats import spearmanr
from scipy.constants import speed_of_light as C
from pyproj import Transformer
from datetime import datetime

try:
    import seaborn as sns
    sns.set_theme(style='whitegrid')
except ImportError:
    pass

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)
    print(f'TF GPU  : {gpus[0].name}')
else:
    print('TF GPU  : NOT detected – running on CPU')
    print('          To fix: check CUDA/cuDNN paths or run:')
    print('          conda install -c conda-forge cudatoolkit cudnn')
tf.get_logger().setLevel('ERROR')
tf.random.set_seed(42)

import sionna
from sionna.rt import load_scene, RadioMaterial, PlanarArray, Transmitter, Receiver

_HAS_OFDM = False
try:
    from sionna.channel import cir_to_ofdm_channel, subcarrier_frequencies
    _HAS_OFDM = True; print('OFDM    : OK  (sionna.channel)')
except (ImportError, AttributeError):
    try:
        from sionna.channel.ofdm import cir_to_ofdm_channel, subcarrier_frequencies
        _HAS_OFDM = True; print('OFDM    : OK  (sionna.channel.ofdm)')
    except: print('OFDM    : NOT found – power fallback will be used')

# ── Mitsuba – try variants in order of preference ─────────────────────────────
# Available in this build: scalar_rgb, scalar_spectral, cuda_ad_rgb, llvm_ad_rgb
# Polarized variants (mono_polarized) not compiled in this build.
_HAS_MI = False
_MI_VARIANT_PREFERENCE = [
    'cuda_ad_rgb',          # GPU + autodiff (best for diff-RT)
    'llvm_ad_rgb',          # CPU + autodiff (fallback)
    'scalar_rgb',           # CPU scalar fallback
]
try:
    import mitsuba as mi
    for _var in _MI_VARIANT_PREFERENCE:
        try:
            mi.set_variant(_var)
            _HAS_MI = True
            print(f'Mitsuba : {mi.variant()}')
            break
        except Exception:
            continue
    if not _HAS_MI:
        print(f'Mitsuba : imported but no usable variant found')
        print(f'          Available: {", ".join(_MI_VARIANT_PREFERENCE)}')
except ImportError:
    print('Mitsuba : NOT installed')

_HAS_RIO = False
try:
    import rasterio as rio; _HAS_RIO = True; print('rasterio: OK')
except ImportError:
    print('rasterio: NOT available')

_HAS_OSM = False
try:
    import osmnx as ox
    from shapely.geometry import box, Polygon, MultiPolygon
    from shapely.ops import unary_union
    _HAS_OSM = True; print('osmnx   : OK')
except ImportError:
    print('osmnx   : NOT available – pip install osmnx shapely')

print(f'Python  : {sys.version.split()[0]}')
print(f'TF      : {tf.__version__}')
print(f'Sionna  : {sionna.__version__}')

def _safe(v):
    if hasattr(v, 'numpy'): return float(v.numpy())
    if hasattr(v, 'item'):  return float(v.item())
    return float(v)

def _to_numpy(t):
    if isinstance(t, tuple): return t[0].numpy() + 1j * t[1].numpy()
    if hasattr(t, 'numpy'): return t.numpy()
    return np.array(t)

def _cm_to_numpy(cm_obj):
    for attr in ('path_gain', 'rss', 'as_tensor'):
        if not hasattr(cm_obj, attr): continue
        val = getattr(cm_obj, attr)
        arr = val() if callable(val) else val
        if hasattr(arr, 'numpy'): return arr.numpy()
        try: return np.array(arr)
        except: pass
    raise AttributeError('Cannot extract path_gain from CoverageMap.')

## CELL 0b · Install Missing Dependencies

Run once if packages are missing.

In [ ]:
# Uncomment and run once, then restart kernel
# import subprocess, sys
# subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet',
#     'osmnx', 'shapely', 'pyvista', 'open3d', 'rasterio', 'pyproj',
#     'ipyleaflet', 'ipyvolume', 'tqdm'])
# print('Done – restart kernel.')

## CELL 0c · City Bounding Box

**Edit this cell to switch cities.** Keep the OSM download area ≤ 4 km × 4 km for speed.

The full Nottingham scene points you provided span ~17 km × 9 km (≈93k buildings).
A cropped city-centre tile is used for OSM download; full bounds are kept for Sionna scene.

In [ ]:
# ── City selector ─────────────────────────────────────────────────────────────
CITY_NAME = 'Nottingham'

# ── Full scene bbox (from sionna_web scene points) ────────────────────────────
# Point 0: (-1.447449, 52.918218)
# Point 1: (-1.447449, 53.001149)
# Point 2: (-1.206779, 53.001149)
# Point 3: (-1.206779, 52.918218)
SCENE_WEST   = -1.447449
SCENE_EAST   = -1.206779
SCENE_SOUTH  =  52.918218
SCENE_NORTH  =  53.001149

# Active scene bounds (full area)
WEST, EAST, SOUTH, NORTH = SCENE_WEST, SCENE_EAST, SCENE_SOUTH, SCENE_NORTH

center_lon = (WEST  + EAST)  / 2
center_lat = (SOUTH + NORTH) / 2

# ── Coordinate system for UK (zone 30N) ──────────────────────────────────────
UTM_EPSG = 32630   # WGS84 / UTM zone 30N  (UK)
BNG_EPSG = 27700   # British National Grid  (for UK DEM TIFFs)

# ── Project paths ─────────────────────────────────────────────────────────────
BASE_DIR  = os.path.expanduser(f'~/Documents/FYP2026/{CITY_NAME.lower()}')
OUT_DIR   = os.path.join(BASE_DIR, 'results_sionna019')
SCENE_DIR = BASE_DIR
os.makedirs(OUT_DIR, exist_ok=True)

# ── Scene rebuild control ────────────────────────────────────────────────────
# Set False to skip CELL 2+3 and load existing scene.xml directly in CELL 4
FORCE_REBUILD_SCENE = False

# DEM TIF – Nottingham terrain
DEM_TIFF    = '/home/georgeskai/Documents/Region/nottingham3602/uk_terrain_nottingham_aoi.tif'

SCENE_XML   = os.path.join(SCENE_DIR, 'scene', 'scene.xml')
TX_CSV      = os.path.join(SCENE_DIR, 'transmitter_positions.csv')
RX_CSV      = os.path.join(SCENE_DIR, 'receiver_locations.csv')
PARAMS_JSON = os.path.join(SCENE_DIR, 'scene_parameters.json')

# ── City-specific building height cap ───────────────────────────────────────
# Adjust per city. Used to reject SRTM/nDSM tree noise.
# Typical values: Nottingham=40, London=200, Paris=150, NYC=400
CITY_MAX_HEIGHT_M = 40.0
CITY_MIN_HEIGHT_M =  2.0

# ── RF link budget (match to Ofcom drive test metadata) ──────────────────────
FREQUENCY_HZ     = 3.6e9    # carrier frequency (Hz)
BANDWIDTH_HZ     = 20e6     # channel bandwidth (Hz)

# TX parameters – set each independently per Ofcom/site data
TX_POWER_DBM     = 49.0     # transmitter output power at antenna port (dBm)
TX_ANTENNA_GAIN  = 21.0     # antenna gain (dBi)  – 0 for omni, 15-21 for sector
TX_CABLE_LOSS    =  0.0     # feeder/cable loss (dB, positive = loss)
TX_BODY_LOSS     =  0.0     # body/penetration loss at TX side (dB)

# RX parameters
RX_ANTENNA_GAIN  =  0.0     # RX antenna gain (dBi) – 0 for drive-test dongle
RX_CABLE_LOSS    =  0.0     # RX feeder loss (dB)
LNA_GAIN_DB      =  0.0     # LNA gain if present (dB)
NOISE_FLOOR      = -120.0   # receiver noise floor (dBm)

# Computed EIRP and system gain
EIRP_DBM  = TX_POWER_DBM + TX_ANTENNA_GAIN - TX_CABLE_LOSS - TX_BODY_LOSS
SYS_GAIN  = RX_ANTENNA_GAIN + LNA_GAIN_DB  - RX_CABLE_LOSS

# Legacy aliases (used elsewhere in notebook)
TX_POWER_DBM_EFF = EIRP_DBM
TX_GAIN_DBI      = TX_ANTENNA_GAIN
RX_GAIN_DBI      = RX_ANTENNA_GAIN

NUM_SUBCARRIERS    = 76
SUBCARRIER_SPACING = 30e3
if _HAS_OFDM:
    FREQUENCIES = subcarrier_frequencies(NUM_SUBCARRIERS, SUBCARRIER_SPACING)

MAX_DEPTH      = 4
NUM_SAMPLES_CM = 500_000
NUM_SAMPLES_PS = 200_000
GRID_SIZE_M    = 20.0

# Height above ground level
TX_AGL_M       = 25.0   # TX antenna height above ground (m)
RX_AGL_M       =  1.5   # RX height above ground (m) – 1.5 = vehicle roof

_tx_w    = 10**((TX_POWER_DBM - 30) / 10)
_noise_w = 10**((NOISE_FLOOR  - 30) / 10)
SNR_SCALE = _tx_w / _noise_w

# ── Scene area size (sanity check) ────────────────────────────────────────────
_gps_to_utm_tmp = Transformer.from_crs('EPSG:4326', f'EPSG:{UTM_EPSG}', always_xy=True)
_sw = _gps_to_utm_tmp.transform(WEST,  SOUTH)
_ne = _gps_to_utm_tmp.transform(EAST,  NORTH)
_w_km = (_ne[0] - _sw[0]) / 1000
_h_km = (_ne[1] - _sw[1]) / 1000

print('=' * 60)
print(f'CITY          : {CITY_NAME}')
print(f'Scene bbox    : lon [{WEST:.6f}, {EAST:.6f}]')
print(f'                lat [{SOUTH:.6f}, {NORTH:.6f}]')
print(f'Area          : {_w_km:.2f} km x {_h_km:.2f} km')
print(f'Center        : ({center_lon:.6f}, {center_lat:.6f})')
print(f'UTM EPSG      : {UTM_EPSG}')
print(f'Frequency     : {FREQUENCY_HZ/1e9:.3f} GHz')
print(f'TX power      : {TX_POWER_DBM:.1f} dBm  +  {TX_ANTENNA_GAIN:.1f} dBi  -  {TX_CABLE_LOSS:.1f} dB loss')
print(f'EIRP          : {EIRP_DBM:.1f} dBm')
print(f'RX gain       : {RX_ANTENNA_GAIN:.1f} dBi  |  System gain: {SYS_GAIN:.1f} dB')
print(f'DEM           : {DEM_TIFF}')
print(f'Output dir    : {OUT_DIR}')
print('=' * 60)

if _w_km > 10 or _h_km > 10:
    print(f'NOTE: Large area ({_w_km:.1f}x{_h_km:.1f} km). OSM download may take ~1 hour.')


## CELL 1 · Coordinate Utilities + DEM Elevation

- `gps_to_local(lon, lat)` → UTM → subtract scene origin → local XY
- `local_to_gps(x, y)` → reverse
- `get_dem_elevation(local_x, local_y)` → rasterio bilinear lookup
- `ray_cast_ground_z(x, y)` → Mitsuba ray intersect for terrain height

In [ ]:
gps_to_utm = Transformer.from_crs('EPSG:4326', f'EPSG:{UTM_EPSG}', always_xy=True)
utm_to_gps = Transformer.from_crs(f'EPSG:{UTM_EPSG}', 'EPSG:4326', always_xy=True)
utm_to_bng = Transformer.from_crs(f'EPSG:{UTM_EPSG}', f'EPSG:{BNG_EPSG}', always_xy=True)

utm_center_x, utm_center_y = gps_to_utm.transform(center_lon, center_lat)
print(f'UTM center : ({utm_center_x:.1f}, {utm_center_y:.1f})')

def gps_to_local(lon, lat, height=0.0):
    ux, uy = gps_to_utm.transform(lon, lat)
    return float(ux - utm_center_x), float(uy - utm_center_y), float(height)

def local_to_gps(x, y):
    lon, lat = utm_to_gps.transform(_safe(x) + utm_center_x, _safe(y) + utm_center_y)
    return float(lon), float(lat)

dem_data = dem_nodata = dem_tf = dem_crs = None
if _HAS_RIO and os.path.exists(DEM_TIFF):
    _src      = rio.open(DEM_TIFF)
    dem_data  = _src.read(1).astype(np.float32)
    dem_nodata= _src.nodata
    dem_tf    = _src.transform
    dem_crs   = str(_src.crs)
    print(f'DEM     : {dem_data.shape}  nodata={dem_nodata}  CRS={dem_crs}')
else:
    print('DEM     : not loaded (rasterio missing or file absent)')

_is_bng_dem = dem_crs is not None and ('27700' in dem_crs or 'OSGB' in dem_crs.upper())

def get_dem_elevation(local_x, local_y):
    if dem_data is None: return 0.0
    utm_x = _safe(local_x) + utm_center_x
    utm_y = _safe(local_y) + utm_center_y
    px, py = (utm_to_bng.transform(utm_x, utm_y) if _is_bng_dem
              else utm_to_gps.transform(utm_x, utm_y))
    col_f, row_f = ~dem_tf * (px, py)
    r, c = int(np.floor(row_f)), int(np.floor(col_f))
    H, W = dem_data.shape
    if 0 <= r < H-1 and 0 <= c < W-1:
        dr, dc = row_f - r, col_f - c
        z = ((1-dr)*(1-dc)*dem_data[r,c]   + (1-dr)*dc*dem_data[r,c+1] +
              dr*(1-dc)*dem_data[r+1,c]    + dr*dc*dem_data[r+1,c+1])
        if dem_nodata is None or not np.isclose(float(z), dem_nodata):
            return float(z)
    return 0.0

def ray_cast_ground_z(x, y, max_height=2000.0):
    if _HAS_MI:
        try:
            ray = mi.Ray3f(mi.Point3f(float(x), float(y), max_height),
                           mi.Vector3f(0.0, 0.0, -1.0))
            si = scene.mi_scene.ray_intersect(ray)
            if si.is_valid():
                z_val = si.p.z
                return float(z_val.item()) if hasattr(z_val, 'item') else float(z_val)
        except Exception: pass
    return get_dem_elevation(x, y)

print('Coordinate utilities ready.')
print(f'  gps_to_local({center_lon:.4f}, {center_lat:.4f}) → {gps_to_local(center_lon, center_lat)[:2]}')

## CELL 2 · OSM Map Download

Downloads **all buildings** from OpenStreetMap for the full scene bbox defined in CELL 0c.

- Full Nottingham (~16 km x 9 km, ~93k buildings): download takes ~45–90 min; cached to GeoJSON after first run.
- On re-run, loads instantly from the cached GeoJSON file.
- Buildings with no height tag default to **10 m**; tagged floors use **3 m/level**.


In [ ]:
if FORCE_REBUILD_SCENE or not os.path.exists(
        os.path.join(OUT_DIR, 'osm_buildings.geojson')):
    pass  # run block below
# ── OSM download (skip if scene already built) ────────────────────────────────
# Set FORCE_REBUILD_SCENE=True in CELL 0c to re-download

import os, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# ── Paths ─────────────────────────────────────────────────────────────────────
OSM_GEOJSON = os.path.join(OUT_DIR, 'osm_buildings.geojson')
OSM_XML     = os.path.join(SCENE_DIR, 'scene', 'scene.xml')
os.makedirs(os.path.dirname(OSM_XML), exist_ok=True)

print(f'OSM download area : lon [{WEST:.5f}, {EAST:.5f}]  lat [{SOUTH:.5f}, {NORTH:.5f}]')
print(f'                  : {_w_km:.2f} km × {_h_km:.2f} km')

if not _HAS_OSM:
    print('\nosmnx not available. Install with:\n  pip install osmnx shapely')
else:
    import geopandas as gpd

    # ── Download or load from cache ───────────────────────────────────────────
    if os.path.exists(OSM_GEOJSON):
        print(f'\nLoading cached buildings from {OSM_GEOJSON} ...')
        gdf = gpd.read_file(OSM_GEOJSON)
        print(f'  Loaded {len(gdf)} buildings from cache.')
    else:
        print('\nDownloading buildings from OpenStreetMap ...')
        t0 = time.time()

        # osmnx 2.0.x API change:
        #   OLD (1.x): ox.features_from_bbox(north=N, south=S, east=E, west=W, tags=tags)
        #   NEW (2.0): ox.features_from_bbox(bbox=(west, south, east, north), tags=tags)
        try:
            ox.settings.log_level = 30   # WARNING only — suppress INFO spam (osmnx 2.0)
        except AttributeError:
            try: ox.settings.log_console = False   # fallback for osmnx 1.x
            except: pass
        ox.settings.timeout = 180

        gdf = ox.features_from_bbox(
            bbox=(WEST, SOUTH, EAST, NORTH),   # osmnx 2.0: (left, bottom, right, top)
            tags={'building': True}
        )
        print(f'  Downloaded {len(gdf)} features in {time.time()-t0:.1f}s')

        # Filter: keep only polygon footprints ≥ 20 m²
        gdf = gdf[gdf.geometry.geom_type.isin(['Polygon', 'MultiPolygon'])].copy()
        gdf = gdf.to_crs('EPSG:4326')
        gdf_utm = gdf.to_crs(f'EPSG:{UTM_EPSG}')
        gdf     = gdf[gdf_utm.geometry.area >= 20.0].reset_index(drop=True)
        print(f'  After polygon + area filter: {len(gdf)} buildings')

        # Cache to GeoJSON: save geometry + useful OSM height/type tags
        _keep_cols = ['geometry']
        for _c in ['height','building:height','building:levels',
                   'building','building:material']:
            if _c in gdf.columns:
                # convert list/complex cells to string to avoid GeoJSON errors
                gdf[_c] = gdf[_c].apply(
                    lambda v: str(v) if isinstance(v, (list, dict)) else v)
                _keep_cols.append(_c)
        gdf[_keep_cols].to_file(OSM_GEOJSON, driver='GeoJSON')
        print(f'  Saved to {OSM_GEOJSON}  (cols: {_keep_cols})')

    # ── Building height extraction ─────────────────────────────────────────────
    # Priority: OSM height tag > building:levels > SRTM nDSM > OSM type heuristic
    LEVEL_HEIGHT_M = 3.0

    # OSM building-type heuristic defaults (floors × 3 m)
    _TYPE_FLOORS = {
        'house': 2, 'detached': 2, 'semidetached_house': 2, 'terrace': 2,
        'bungalow': 1, 'static_caravan': 1, 'shed': 1, 'garage': 1, 'garages': 1,
        'residential': 3, 'apartments': 5, 'block_of_flats': 5,
        'commercial': 3, 'retail': 2, 'supermarket': 2, 'kiosk': 1,
        'office': 6, 'hotel': 7,
        'industrial': 2, 'warehouse': 1, 'factory': 2,
        'church': 6, 'cathedral': 8, 'chapel': 4, 'mosque': 4, 'temple': 3,
        'school': 3, 'university': 4, 'hospital': 5,
        'train_station': 4, 'transportation': 3,
        'stadium': 8, 'grandstand': 5,
        'yes': 3,   # generic tagged building
    }



    def _parse_height(row):
        # 1. Explicit OSM height tag
        for col in ('height', 'building:height'):
            if col in row and pd.notna(row[col]):
                try: return float(str(row[col]).split()[0])
                except: pass
        # 2. Building levels tag
        if 'building:levels' in row and pd.notna(row.get('building:levels')):
            try: return float(row['building:levels']) * LEVEL_HEIGHT_M
            except: pass
        # 3. OSM building-type heuristic
        btype = str(row.get('building', '')).strip().lower()
        if btype in _TYPE_FLOORS:
            return _TYPE_FLOORS[btype] * LEVEL_HEIGHT_M
        return 8.0   # final fallback

    heights = gdf.apply(_parse_height, axis=1).values
    n_osm    = int(sum(1 for _, r in gdf.iterrows()
                       for c in ('height','building:height')
                       if c in r and pd.notna(r[c])))
    n_levels = int(sum(1 for _, r in gdf.iterrows()
                       if 'building:levels' in r and pd.notna(r.get('building:levels'))))
    print(f'\n  Building heights: min={heights.min():.1f}  mean={heights.mean():.1f}  '
          f'max={heights.max():.1f} m')
    # Clip to city-configured height range (set CITY_MAX_HEIGHT_M in CELL 0c)
    heights = np.clip(heights, CITY_MIN_HEIGHT_M, CITY_MAX_HEIGHT_M)

    print(f'  Sources: OSM tag={n_osm}  levels={n_levels}  '
          f'nDSM+heuristic={len(gdf)-n_osm-n_levels}')
    print(f'  Height stats: min={heights.min():.1f}  mean={heights.mean():.1f}  '
          f'max={heights.max():.1f} m  |  cap [{CITY_MIN_HEIGHT_M}, {CITY_MAX_HEIGHT_M}] m')

    # ── Plot ───────────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    # Map — colour by height
    gdf.plot(ax=axes[0], column=heights, cmap='YlOrRd',
             legend=True, legend_kwds={'label': 'Height (m)'},
             edgecolor='k', linewidth=0.2, alpha=0.8)
    axes[0].add_patch(Rectangle((WEST, SOUTH), EAST-WEST, NORTH-SOUTH,
                                 fill=False, edgecolor='blue', lw=2, label='OSM bbox'))
    axes[0].set_title(f'{CITY_NAME} – OSM Buildings ({len(gdf):,})')
    axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude')
    axes[0].legend()

    # Height histogram
    axes[1].hist(heights, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    axes[1].axvline(8.0, color='red', ls='--', label='Final fallback (8 m)')
    axes[1].set_xlabel('Building Height (m)'); axes[1].set_ylabel('Count')
    axes[1].set_title('Building Height Distribution'); axes[1].legend()

    plt.suptitle(f'{CITY_NAME} OSM Download  |  {len(gdf):,} buildings', fontsize=13)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, 'osm_buildings.png'), dpi=150)
    plt.show()
    print(f'Map saved → {os.path.join(OUT_DIR, "osm_buildings.png")}')

## CELL 3 · Generate Mitsuba 3 Scene XML from OSM Buildings

Converts OSM footprints + heights into a Sionna 0.19-compatible Mitsuba 3 scene XML.
Each building is extruded as a box mesh. Ground plane is added automatically.

> Skip this cell if you already have a `scene.xml` from sionna_web.

In [ ]:
# OSM -> Mitsuba 3 scene: PLY meshes + scene.xml (Sionna 0.19 compatible)
# Skipped automatically if scene.xml exists and FORCE_REBUILD_SCENE=False

_scene_xml_exists = os.path.exists(SCENE_XML)
if not FORCE_REBUILD_SCENE and _scene_xml_exists:
    print(f'Scene already exists: {SCENE_XML}')
    print('Set FORCE_REBUILD_SCENE=True in CELL 0c to regenerate.')
    XML_OK = True
else:
    # Buildings grouped by material -> one PLY per material type.
    # Terrain sampled from DEM -> terrain.ply
    # All PLY files in scene/meshes/; scene.xml references them by filename.

    import xml.etree.ElementTree as ET
    import xml.dom.minidom as minidom
    import numpy as np
    import struct, os, time

    _HAS_DEM_RASTERIO = False
    try:
        import rasterio
        _HAS_DEM_RASTERIO = True
    except ImportError:
        pass

    if not _HAS_OSM or 'gdf' not in dir():
        print('Run CELL 2 first.')
    else:
        print(f'Generating Mitsuba 3 scene (PLY + XML) from {len(gdf)} buildings ...')
        t0 = time.time()

        SCENE_OUT_DIR  = os.path.join(BASE_DIR, 'scene')
        MESHES_DIR     = os.path.join(SCENE_OUT_DIR, 'meshes')
        os.makedirs(MESHES_DIR, exist_ok=True)
        SCENE_XML_OUT  = os.path.join(SCENE_OUT_DIR, 'scene.xml')

        # ── Load DEM ──────────────────────────────────────────────────────────
        _dem_ds2 = _dem_data2 = _dem_tf2 = _wgs_to_dem2 = None
        if _HAS_DEM_RASTERIO and os.path.exists(DEM_TIFF):
            try:
                from pyproj import Transformer as _Tr2
                _dem_ds2   = rasterio.open(DEM_TIFF)
                _dem_data2 = _dem_ds2.read(1).astype(np.float32)
                _dem_tf2   = _dem_ds2.transform
                _epsg2     = str(_dem_ds2.crs.to_epsg()) if _dem_ds2.crs else ''
                _wgs_to_dem2 = _Tr2.from_crs('EPSG:4326',
                                   f'EPSG:{_epsg2}' if _epsg2 else _dem_ds2.crs,
                                   always_xy=True)
                print(f'  DEM: {_dem_data2.shape} px, EPSG:{_epsg2}')
            except Exception as _e:
                print(f'  DEM load failed: {_e}')

        def _dem_z2(lon, lat):
            if _dem_data2 is None: return 0.0
            try:
                dx, dy = _wgs_to_dem2.transform(lon, lat)
                tf = _dem_tf2
                col = (dx - tf.c) / tf.a
                row = (dy - tf.f) / tf.e
                r, c = int(round(row)), int(round(col))
                nr, nc = _dem_data2.shape
                r = max(0, min(nr-1, r)); c = max(0, min(nc-1, c))
                v = float(_dem_data2[r, c])
                return v if (np.isfinite(v) and v > -9000) else 0.0
            except Exception: return 0.0

        origin_elev2 = _dem_z2(center_lon, center_lat)

        # ── PLY writer ────────────────────────────────────────────────────────
        def write_ply(path, verts, faces):
            verts = np.asarray(verts, dtype=np.float32)
            faces = np.asarray(faces, dtype=np.int32)
            with open(path, 'wb') as fp:
                hdr = (
                    'ply\nformat binary_little_endian 1.0\n'
                    f'element vertex {len(verts)}\n'
                    'property float x\nproperty float y\nproperty float z\n'
                    f'element face {len(faces)}\n'
                    'property list uchar int vertex_indices\n'
                    'end_header\n'
                )
                fp.write(hdr.encode())
                fp.write(verts.tobytes())
                for tri in faces:
                    fp.write(struct.pack('<B', 3))
                    fp.write(struct.pack('<3i', *tri))

        # ── Project OSM to UTM ────────────────────────────────────────────────
        gdf_utm = gdf.to_crs(f'EPSG:{UTM_EPSG}')
        ox_utm, oy_utm = utm_center_x, utm_center_y

        ITU_MATS = ['itu_concrete','itu_brick','itu_glass','itu_wood']

        def _bld_mat(row):
            mat = str(row.get('building:material','')).lower()
            tag = str(row.get('building','')).lower()
            if 'glass' in mat:                      return 'itu_glass'
            if 'wood' in mat or 'timber' in mat:    return 'itu_wood'
            if 'brick' in mat:                      return 'itu_brick'
            if tag in ('greenhouse','glasshouse'):  return 'itu_glass'
            return 'itu_concrete'

        # Collect per-material vertex/face lists
        mat_verts = {m: [] for m in ITU_MATS}
        mat_faces = {m: [] for m in ITU_MATS}

        gdf_wgs_list  = list(gdf.iterrows())
        gdf_utm_list  = list(gdf_utm.iterrows())
        skipped = 0

        for idx in range(len(gdf_utm_list)):
            _, row_utm = gdf_utm_list[idx]
            _, row_wgs = gdf_wgs_list[idx]
            geom = row_utm.geometry
            if geom is None or geom.is_empty:
                skipped += 1; continue
            try:
                hull = geom.convex_hull
                if hull.geom_type != 'Polygon': skipped += 1; continue
                ring = list(hull.exterior.coords[:-1])
            except Exception:
                skipped += 1; continue
            n = len(ring)
            if n < 3: skipped += 1; continue

            # building height
            h = heights[idx] if idx < len(heights) else 8.0

            # base elevation from DEM
            try:
                ctr = row_wgs.geometry.centroid
                base_z = _dem_z2(ctr.x, ctr.y) - origin_elev2
            except Exception:
                base_z = 0.0

            pts = [(x - ox_utm, y - oy_utm) for x, y in ring]
            z0, z1 = base_z, base_z + h

            mat = _bld_mat(dict(row_wgs))
            vl  = mat_verts[mat]
            fl  = mat_faces[mat]
            base_idx = len(vl)

            # bottom ring + top ring
            for x, y in pts: vl.append((x, y, z0))
            for x, y in pts: vl.append((x, y, z1))

            # side walls
            for i in range(n):
                j = (i+1) % n
                b, t = base_idx, base_idx + n
                fl.append((b+i, b+j, t+j))
                fl.append((b+i, t+j, t+i))
            # top cap fan
            for i in range(1, n-1):
                fl.append((base_idx+n, base_idx+n+i, base_idx+n+i+1))
            # bottom cap fan (reversed)
            for i in range(1, n-1):
                fl.append((base_idx, base_idx+i+1, base_idx+i))

        print(f'  Built geometry: {sum(len(v) for v in mat_verts.values()):,} verts'
              f'  {sum(len(f) for f in mat_faces.values()):,} faces  ({skipped} skipped)')

        # ── Write building PLY files ───────────────────────────────────────────
        ply_files = {}
        for mat in ITU_MATS:
            if not mat_verts[mat]: continue
            ply_path = os.path.join(MESHES_DIR, f'{mat}.ply')
            write_ply(ply_path, mat_verts[mat], mat_faces[mat])
            ply_files[mat] = ply_path
            print(f'  {mat}.ply  {len(mat_verts[mat]):,} verts  {len(mat_faces[mat]):,} faces')

        # ── Terrain PLY (DEM 64x64 grid) ──────────────────────────────────────
        DEM_GRID_N = 64
        lons_g = np.linspace(WEST,  EAST,  DEM_GRID_N)
        lats_g = np.linspace(SOUTH, NORTH, DEM_GRID_N)
        t_verts, t_faces = [], []
        for lat in lats_g:
            for lon in lons_g:
                ux, uy = gps_to_utm.transform(lon, lat)
                lx, ly = ux - ox_utm, uy - oy_utm
                z = _dem_z2(lon, lat) - origin_elev2
                t_verts.append((lx, ly, z))
        for r in range(DEM_GRID_N-1):
            for c in range(DEM_GRID_N-1):
                i00 = r*DEM_GRID_N + c
                i10 = i00 + 1
                i01 = i00 + DEM_GRID_N
                i11 = i01 + 1
                t_faces += [(i00,i10,i11), (i00,i11,i01)]
        terrain_ply = os.path.join(MESHES_DIR, 'terrain.ply')
        write_ply(terrain_ply, t_verts, t_faces)
        print(f'  terrain.ply  {len(t_verts)} verts  {len(t_faces)} faces')

        # ── scene.xml ─────────────────────────────────────────────────────────
        root = ET.Element('scene', version='3.0.0')
        for name, val in [
            ('scenegen_min_lon',    str(WEST)),
            ('scenegen_max_lon',    str(EAST)),
            ('scenegen_min_lat',    str(SOUTH)),
            ('scenegen_max_lat',    str(NORTH)),
            ('scenegen_origin_lon', str(center_lon)),
            ('scenegen_origin_lat', str(center_lat)),
        ]:
            ET.SubElement(root, 'default', name=name, value=val)

        integ = ET.SubElement(root, 'integrator', type='path')
        ET.SubElement(integ, 'integer', name='max_depth', value='8')

        # Mitsuba BSDF: optical only (no EM props here)
        # ITU-R EM properties (er, sigma) are assigned in CELL 5 via RadioMaterial
        ITU_MAT_IDS = ['itu_concrete','itu_brick','itu_glass',
                       'itu_wood','itu_medium_dry_ground']
        for mat_id in ITU_MAT_IDS:
            bsdf = ET.SubElement(root, 'bsdf', type='diffuse', id=mat_id)
            ET.SubElement(bsdf, 'rgb', name='reflectance', value='0.5 0.5 0.5')

        # terrain shape
        s = ET.SubElement(root, 'shape', type='ply', id='terrain')
        ET.SubElement(s, 'string', name='filename',
                      value=os.path.relpath(terrain_ply, SCENE_OUT_DIR))
        ET.SubElement(s, 'ref', id='itu_medium_dry_ground')

        # building shapes
        for mat, ply_path in ply_files.items():
            s = ET.SubElement(root, 'shape', type='ply', id=mat.replace('itu_','')+'_buildings')
            ET.SubElement(s, 'string', name='filename',
                          value=os.path.relpath(ply_path, SCENE_OUT_DIR))
            ET.SubElement(s, 'ref', id=mat)

        xml_str = minidom.parseString(
            ET.tostring(root, encoding='unicode')
        ).toprettyxml(indent='  ', encoding=None)
        with open(SCENE_XML_OUT, 'w', encoding='utf-8') as f:
            f.write(xml_str)

        SCENE_XML = SCENE_XML_OUT
        XML_OK    = True
        if _dem_ds2: _dem_ds2.close()

        print(f'\n  Done in {time.time()-t0:.1f}s')
        print(f'  scene/meshes/ -> {len(ply_files)+1} PLY files')
        print(f'  scene.xml     -> {SCENE_XML_OUT}')
        print('Ready for CELL 4: load_scene(SCENE_XML)')


# Redefine ray_cast_ground_z with live scene (CELL 1 ran before scene was loaded)
def ray_cast_ground_z(x, y, max_height=2000.0):
    try:
        ray = mi.Ray3f(mi.Point3f(float(x), float(y), max_height),
                       mi.Vector3f(0.0, 0.0, -1.0))
        si = scene.mi_scene.ray_intersect(ray)
        if si.is_valid():
            z_val = si.p.z
            return float(z_val.item()) if hasattr(z_val, 'item') else float(z_val)
    except Exception:
        pass
    return get_dem_elevation(x, y)

print('ray_cast_ground_z redefined with live scene.')

## CELL 4 · Load 3-D Scene & Configure Antennas

**Sionna 0.19 API:** `load_scene(path)` — no `merge_shapes` argument.

In [ ]:
XML_OK = os.path.exists(SCENE_XML)
if not XML_OK:
    raise RuntimeError(
        f'scene.xml not found at {SCENE_XML}\n'
        'Either:\n'
        '  A) Run CELL 3 to generate from OSM (requires Blender for full mesh)\n'
        '  B) Generate via sionna_web Steps 1-3\n'
        '  C) Use Blender + BlenderOSM plugin')

print(f'Loading scene from {SCENE_XML} ...')
scene = load_scene(SCENE_XML)   # Sionna 0.19: NO merge_shapes argument
scene.frequency = FREQUENCY_HZ

scene.tx_array = PlanarArray(
    num_rows=1, num_cols=1,
    vertical_spacing=0.5, horizontal_spacing=0.5,
    pattern='iso', polarization='V')
scene.rx_array = PlanarArray(
    num_rows=1, num_cols=1,
    vertical_spacing=0.5, horizontal_spacing=0.5,
    pattern='iso', polarization='V')

print(f'Scene loaded  : {len(scene.objects)} objects,  {len(scene.radio_materials)} materials')
print(f'Frequency     : {FREQUENCY_HZ/1e9:.3f} GHz')
print(f'Materials     : {list(scene.radio_materials.keys())}')

In [ ]:
# ====================================================================
# CELL 4b — SCENE PREVIEW  [Sionna 0.19]
# ====================================================================
# Requires: conda install -c conda-forge pythreejs ipywidgets
#           jupyter nbextension enable --py widgetsnbextension
#           jupyter nbextension enable --py pythreejs
# Restart the kernel after installing.
# ====================================================================

try:
    import pythreejs  # noqa
    _HAS_PYTHREEJS = True
except ImportError:
    _HAS_PYTHREEJS = False
    print('pythreejs not installed.')
    print('Run in terminal (sionna019 env):')
    print('  conda install -c conda-forge pythreejs ipywidgets -y')
    print('  jupyter nbextension enable --py widgetsnbextension')
    print('  jupyter nbextension enable --py pythreejs')
    print('Then restart kernel and re-run this cell.')

if _HAS_PYTHREEJS:
    print('Launching scene preview ...')
    print('(rotate: left-drag | zoom: scroll | pan: right-drag)')
    # Show scene geometry only (no paths)
    scene.preview()


## CELL 5 · Assign ITU-R P.2040-2 Material Properties

In [ ]:
_ITU_DB = {
    'concrete'          : (5.24,  0.130, 0.40, 0.20),
    'brick'             : (3.91,  0.024, 0.30, 0.20),
    'wood'              : (1.99,  0.005, 0.25, 0.30),
    'glass'             : (6.27,  0.012, 0.08, 0.10),
    'metal'             : (1.00,  1e7,   0.05, 0.10),
    'asphalt'           : (3.00,  0.010, 0.35, 0.20),
    'vegetation'        : (1.30,  0.001, 0.75, 0.05),
    'water'             : (81.0,  0.500, 0.02, 0.05),
    'wet_ground'        : (30.0,  0.150, 0.20, 0.20),
    'medium_dry_ground' : (15.0,  0.035, 0.18, 0.20),
    'very_dry_ground'   : (3.00,  0.001, 0.12, 0.20),
    'marble'            : (7.07,  0.020, 0.08, 0.10),
    'plasterboard'      : (2.73,  0.010, 0.12, 0.20),
}
_DEFAULT_MAT = (4.0, 0.08, 0.30, 0.15)

def _match_itu(mat_name):
    n = mat_name.lower().replace('itu_', '').replace('mat-', '').replace('mat_', '')
    for key in _ITU_DB:
        if key in n: return key
    for key in _ITU_DB:
        if any(part in n for part in key.split('_')): return key
    return None

print('ASSIGNING ITU-R MATERIAL PROPERTIES')
for mat_name, mat in scene.radio_materials.items():
    key = _match_itu(mat_name)
    eps_r, sigma, S, xpd = _ITU_DB.get(key, _DEFAULT_MAT)
    try: mat.relative_permittivity = eps_r
    except: pass
    try: mat.conductivity = sigma
    except: pass
    for a_ in ('scattering_coefficient', 'scattering_coeff'):
        if hasattr(mat, a_):
            try: setattr(mat, a_, S); break
            except: pass
    print(f'  {mat_name:<32}  matched={key or "DEFAULT":<20}  eps={eps_r:.2f}  σ={sigma:.4g}')
print('Done.')

## CELL 6 · Load Transmitter (GPS → local XY, ray-cast Z)

In [ ]:
import time as _time

# ── [1/4] Clear previous transmitters ────────────────────────────────────────
print('[1/4] Clearing previous transmitters ...')
for nm in list(scene.transmitters.keys()):
    scene.remove(nm)
print('  ✓ Cleared')

# ── [2/4] Load TX CSV or use scene centre ─────────────────────────────────────
print('\n[2/4] Loading transmitter ...')
transmitters = []
_t0 = _time.time()

if os.path.exists(TX_CSV):
    df_tx = pd.read_csv(TX_CSV)
    print(f'  ✓ Loaded {len(df_tx)} TX from {TX_CSV}')
    for i, row in df_tx.iterrows():
        lon    = float(row['lon']); lat = float(row['lat'])
        tx_agl = float(row.get('height', TX_AGL_M))
        x, y, _ = gps_to_local(lon, lat)
        ground_z = ray_cast_ground_z(x, y)
        z  = ground_z + tx_agl
        nm = str(row.get('name', f'tx{i:04d}'))
        tx = Transmitter(name=nm, position=(float(x), float(y), float(z)))
        scene.add(tx); transmitters.append(tx)
else:
    print(f'  ⚠ TX CSV not found – using scene centre')
    x, y = 0.0, 0.0
    ground_z = ray_cast_ground_z(x, y)
    z  = ground_z + TX_AGL_M
    tx = Transmitter(name='tx_center', position=(float(x), float(y), float(z)))
    scene.add(tx); transmitters.append(tx)

# Reference TX for downstream cells
tx     = transmitters[0]
abs_z  = _safe(tx.position[2])
tx_agl = TX_AGL_M

# ── [3/4] Antenna arrays ──────────────────────────────────────────────────────
print('\n[3/4] Configuring antenna arrays ...')
scene.tx_array = PlanarArray(
    num_rows=1, num_cols=1,
    vertical_spacing=0.5, horizontal_spacing=0.5,
    pattern='iso', polarization='V')
scene.rx_array = PlanarArray(
    num_rows=1, num_cols=1,
    vertical_spacing=0.5, horizontal_spacing=0.5,
    pattern='iso', polarization='V')
print('  ✓ TX: 1x1 isotropic V-pol')
print('  ✓ RX: 1x1 isotropic V-pol')

# ── [4/4] Summary ─────────────────────────────────────────────────────────────
print('\n[4/4] Transmitter summary:')
for t in transmitters:
    x  = _safe(t.position[0])
    y  = _safe(t.position[1])
    z  = _safe(t.position[2])
    lon, lat = local_to_gps(x, y)
    print(f'  ✓ TX "{t.name}"  GPS=({lon:.5f},{lat:.5f})  '
          f'XY=({x:.1f},{y:.1f})  Z={z:.2f} m  '
          f'AGL={TX_AGL_M:.1f} m  EIRP={EIRP_DBM:.1f} dBm')
print(f'\n  Done in {_time.time()-_t0:.2f}s')


## CELL 7 · Load Receivers (GPS → local XY, ray-cast Z)

In [ ]:
import time as _time

# ── [1/5] Load CSV ───────────────────────────────────────────────────────────
print('[1/5] Loading receiver CSV ...')
if not os.path.exists(RX_CSV):
    print(f'  ✗ RX CSV not found at {RX_CSV}')
    df_rx = None
else:
    df_rx = pd.read_csv(RX_CSV)
    print(f'  ✓ Loaded {len(df_rx)} receivers')

# ── [2/5] Clear previous receivers ───────────────────────────────────────────
print('\n[2/5] Clearing previous receivers ...')
for nm in list(scene.receivers.keys()):
    scene.remove(nm)
print(f'  ✓ Cleared')

# ── [3/5] Convert coordinates ─────────────────────────────────────────────────
print('\n[3/5] Converting coordinates and computing ground heights ...')
receivers = []
_t0 = _time.time()

if df_rx is not None:
    for i, row in df_rx.iterrows():
        lon = float(row['lon']); lat = float(row['lat'])
        agl = float(row.get('height', RX_AGL_M))
        x, y, _ = gps_to_local(lon, lat)
        ground_z = ray_cast_ground_z(x, y)
        z = ground_z + agl
        nm = str(row.get('name', f'RX_{i+1:04d}'))
        rx = Receiver(name=nm, position=(float(x), float(y), float(z)))
        rx._ground_z = ground_z
        rx._agl      = agl
        scene.add(rx)
        receivers.append(rx)
    print(f'  ✓ Loaded {len(receivers)} receivers in {_time.time()-_t0:.2f}s')
else:
    rx = Receiver(name='rx0', position=(100.0, 0.0, RX_AGL_M))
    rx._ground_z = 0.0; rx._agl = RX_AGL_M
    scene.add(rx); receivers.append(rx)
    print('  ✓ Default single receiver placed')

# ── [4/5] Validation ─────────────────────────────────────────────────────────
print('\n[4/5] Validation (first 5 receivers):')
for rx in receivers[:5]:
    x  = _safe(rx.position[0])
    y  = _safe(rx.position[1])
    z  = _safe(rx.position[2])
    gz = getattr(rx, '_ground_z', z - getattr(rx, '_agl', RX_AGL_M))
    lon, lat = local_to_gps(x, y)
    print(f'  {rx.name}: XY({x:.1f}, {y:.1f})  Z={z:.2f}  '
          f'(ground={gz:.2f})  GPS=({lon:.5f},{lat:.5f})')

# ── [5/5] Bbox containment ────────────────────────────────────────────────────
print('\n[5/5] Bbox containment:')
try:
    _bbox  = scene.mi_scene.bbox()
    xs = [_safe(r.position[0]) for r in receivers]
    ys = [_safe(r.position[1]) for r in receivers]
    zs = [_safe(r.position[2]) for r in receivers]
    x_ok = all(float(_bbox.min[0]) <= x <= float(_bbox.max[0]) for x in xs)
    y_ok = all(float(_bbox.min[1]) <= y <= float(_bbox.max[1]) for y in ys)
    z_ok = all(float(_bbox.min[2]) <= z <= float(_bbox.max[2]) for z in zs)
    print(f'  All X inside: {x_ok},  Y inside: {y_ok},  Z inside: {z_ok}')
    if not (x_ok and y_ok):
        print('  ⚠ Some receivers outside scene bbox — check GPS coordinates')
except Exception as _e:
    print(f'  Bbox check skipped: {_e}')

print(f'\nScene: {len(scene.transmitters)} TX,  {len(scene.receivers)} RX')


## CELL 8 · Coverage Map (Sionna 0.19)

In [ ]:
# ====================================================================
# CELL 8 — COVERAGE MAP (WITH & WITHOUT SCATTER)  [Sionna 0.19]
# ====================================================================
import numpy as np, os, time

print('=' * 70)
print('CELL 8 — COVERAGE MAP SOLVER')
print('=' * 70)

bbox   = scene.mi_scene.bbox()
margin = 50.0
gx_min = float(bbox.min[0]) - margin
gx_max = float(bbox.max[0]) + margin
gy_min = float(bbox.min[1]) - margin
gy_max = float(bbox.max[1]) + margin
center_x = (gx_min + gx_max) / 2
center_y = (gy_min + gy_max) / 2
ground_z_at_center = ray_cast_ground_z(center_x, center_y)
center_z = ground_z_at_center + RX_AGL_M
nx = int((gx_max - gx_min) / GRID_SIZE_M)
ny = int((gy_max - gy_min) / GRID_SIZE_M)
print(f'Grid: {nx} x {ny} cells  ({GRID_SIZE_M} m res)  map Z={center_z:.2f} m')

# ── Scene sanity check ───────────────────────────────────────────────────────
print(f'\n[DIAG] TX count      : {len(scene.transmitters)}')
for _n, _t in scene.transmitters.items():
    print(f'       TX "{_n}"  pos={[round(_safe(_t.position[i]),2) for i in range(3)]}')
print(f'[DIAG] CM center_z   : {center_z:.2f} m')
print(f'[DIAG] Grid          : {nx} x {ny}  gx=[{gx_min:.0f},{gx_max:.0f}]  gy=[{gy_min:.0f},{gy_max:.0f}]')

# ── Helper: path_gain -> (rssi_dBm, path_loss_dB) ────────────────────────────
def _cm_to_dbm(cm):
    """Return (rssi_db, path_loss_db) arrays from a Sionna 0.19 CoverageMap."""
    arr = np.array(cm.path_gain)
    if arr.ndim == 3:   arr = arr[0]       # (num_tx, ny, nx) -> (ny, nx)
    elif arr.ndim == 4: arr = arr[0, 0]    # edge-case extra dim
    path_loss_db = -10.0 * np.log10(np.maximum(arr, 1e-20))
    # Use EIRP_DBM directly — tx.power_dbm is Sionna internal default, not our EIRP
    rssi_db = EIRP_DBM - path_loss_db + RX_GAIN_DBI + LNA_GAIN_DB
    return rssi_db, path_loss_db

# ── WITH scattering ───────────────────────────────────────────────────────────
print('\nComputing coverage map WITH scattering ...')
t0 = time.time()
cm_scatter = scene.coverage_map(
    cm_center=(center_x, center_y, center_z),
    cm_orientation=(0., 0., 0.),
    cm_size=(gx_max - gx_min, gy_max - gy_min),
    cm_cell_size=(GRID_SIZE_M, GRID_SIZE_M),
    max_depth=MAX_DEPTH, num_samples=NUM_SAMPLES_CM,
    los=True, reflection=True, scattering=True, diffraction=False)
print(f'  Done in {time.time()-t0:.1f}s')

# ── WITHOUT scattering ────────────────────────────────────────────────────────
print('\nComputing coverage map WITHOUT scattering ...')
t0 = time.time()
cm_no_scatter = scene.coverage_map(
    cm_center=(center_x, center_y, center_z),
    cm_orientation=(0., 0., 0.),
    cm_size=(gx_max - gx_min, gy_max - gy_min),
    cm_cell_size=(GRID_SIZE_M, GRID_SIZE_M),
    max_depth=MAX_DEPTH, num_samples=NUM_SAMPLES_CM,
    los=True, reflection=True, scattering=False, diffraction=False)
print(f'  Done in {time.time()-t0:.1f}s')

# ── Decode to numpy ──────────────────────────────────────────────────────────
rssi_scatter,    path_loss_scatter    = _cm_to_dbm(cm_scatter)
_raw_pg = np.array(cm_scatter.path_gain)
print(f'[DIAG] path_gain shape: {_raw_pg.shape}  dtype={_raw_pg.dtype}')
print(f'[DIAG] path_gain raw: min={_raw_pg.min():.2e}  max={_raw_pg.max():.2e}  mean={_raw_pg.mean():.2e}')
print(f'[DIAG] non-zero cells: {(_raw_pg > 1e-30).sum()} / {_raw_pg.size}')
rssi_no_scatter, path_loss_no_scatter = _cm_to_dbm(cm_no_scatter)

NOISE_FLOOR_DBM = -120.0   # filter uncovered (zero-energy) cells for stats
for label, r, pl in [
    ('With scatter',    rssi_scatter,    path_loss_scatter),
    ('Without scatter', rssi_no_scatter, path_loss_no_scatter),
]:
    covered = r > NOISE_FLOOR_DBM          # cells with meaningful signal
    n_cov   = int(covered.sum())
    n_total = int(r.size)
    v  = r[covered]
    p  = pl[covered]
    print(f'{label:20s}')
    print(f'  Coverage: {n_cov:,}/{n_total:,} cells ({100*n_cov/n_total:.1f}%)')
    if n_cov:
        print(f'  RSSI (covered): mean={v.mean():.1f}  std={v.std():.1f}  '
              f'min={v.min():.1f}  max={v.max():.1f} dBm')
        print(f'  PL   (covered): mean={p.mean():.1f}  std={p.std():.1f}  '
              f'min={p.min():.1f}  max={p.max():.1f} dB')
    else:
        print(f'  ⚠ No covered cells — increase NUM_SAMPLES_CM in CELL 0c')

# Scatter impact on covered cells
cov_both = (rssi_scatter > NOISE_FLOOR_DBM) & (rssi_no_scatter > NOISE_FLOOR_DBM)
if cov_both.sum() > 0:
    delta = rssi_scatter[cov_both] - rssi_no_scatter[cov_both]
    print(f'\nScatter impact (covered cells): '
          f'mean={delta.mean():.2f}  std={delta.std():.2f}  '
          f'max={delta.max():.2f} dB')
    print(f'  (near-zero mean = NUM_SAMPLES_CM too low; increase to 2M+ to see scatter effect)')


## CELL 9 · Interpolate to Receiver Positions

In [ ]:
# ====================================================================
# CELL 9 — INTERPOLATE COVERAGE MAP TO RECEIVER POSITIONS
# ====================================================================
from scipy.spatial import KDTree
import pandas as pd, numpy as np, os

print('=' * 70)
print('CELL 9 — INTERPOLATING RADIO MAPS TO RX POSITIONS')
print('=' * 70)

# Grid centres (must match CELL 8 grid)
x_centers = np.linspace(gx_min, gx_max, nx)
y_centers  = np.linspace(gy_min, gy_max, ny)
X, Y = np.meshgrid(x_centers, y_centers)
grid_points = np.column_stack([X.ravel(), Y.ravel()])
tree = KDTree(grid_points)

# Receiver XY positions (local UTM, metres)
rx_coords = np.array([(_safe(rx.position[0]), _safe(rx.position[1])) for rx in receivers])
distances, indices = tree.query(rx_coords)

def interpolate_and_save(rssi_map, path_loss_map, suffix):
    rssi_at_rx = rssi_map.ravel()[indices]
    pl_at_rx   = path_loss_map.ravel()[indices]
    # Convert to GPS for RMSE comparison with Ofcom drive test CSV
    lon_lat    = [local_to_gps(x, y) for x, y in rx_coords]
    df = pd.DataFrame({
        'receiver':     [rx.name for rx in receivers],
        'lon':          [ll[0] for ll in lon_lat],
        'lat':          [ll[1] for ll in lon_lat],
        'x_m':          rx_coords[:, 0],
        'y_m':          rx_coords[:, 1],
        'z_m':          [_safe(rx.position[2]) for rx in receivers],
        'rssi_dbm':     rssi_at_rx,
        'path_loss_db': pl_at_rx,
        'grid_dist_m':  distances,
    })
    out = os.path.join(OUT_DIR, f'receiver_results_{suffix}.csv')
    df.to_csv(out, index=False)
    print(f'  Saved {len(df)} receivers -> {out}')
    return df

df_scatter    = interpolate_and_save(rssi_scatter,    path_loss_scatter,    'with_scatter')
df_no_scatter = interpolate_and_save(rssi_no_scatter, path_loss_no_scatter, 'without_scatter')

for label, df in [('With scatter', df_scatter), ('Without scatter', df_no_scatter)]:
    print(f'{label:20s}: RSSI mean={df.rssi_dbm.mean():.1f}  std={df.rssi_dbm.std():.1f} dBm  '
          f'| PL  mean={df.path_loss_db.mean():.1f}  std={df.path_loss_db.std():.1f} dB')


## CELL 10 · Coverage Map Visualisation

In [ ]:
# ====================================================================
# CELL 10 — COVERAGE MAPS (WITH vs WITHOUT SCATTER)
# ====================================================================
import matplotlib.pyplot as plt, os

print('=' * 70)
print('CELL 10 — PLOTTING COVERAGE MAPS')
print('=' * 70)

tx   = list(scene.transmitters.values())[0]
tx_x = _safe(tx.position[0]); tx_y = _safe(tx.position[1])

# Sample receivers for overlay (every 200th for readability)
sample_step = max(1, len(receivers) // 200)
rx_plot_x = [_safe(r.position[0]) for r in receivers[::sample_step]]
rx_plot_y = [_safe(r.position[1]) for r in receivers[::sample_step]]

vmin, vmax = -120, -40
ext = [gx_min, gx_max, gy_min, gy_max]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, data, title in [
    (axes[0], rssi_scatter,    f'With scattering @ {FREQUENCY_HZ/1e9:.2f} GHz'),
    (axes[1], rssi_no_scatter, f'Without scattering @ {FREQUENCY_HZ/1e9:.2f} GHz'),
]:
    im = ax.imshow(data, origin='lower', extent=ext,
                   cmap='jet', aspect='auto', vmin=vmin, vmax=vmax)
    ax.scatter(tx_x, tx_y, marker='*', s=300, c='gold', edgecolors='black', label='TX')
    ax.scatter(rx_plot_x, rx_plot_y, s=5, c='cyan', alpha=0.5, label='RX')
    ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)')
    ax.set_title(title); ax.legend(loc='upper right', fontsize=8)
    plt.colorbar(im, ax=ax, label='RSSI (dBm)')

plt.suptitle(f'Coverage Map | EIRP={EIRP_DBM:.1f} dBm | {FREQUENCY_HZ/1e9:.2f} GHz', fontsize=13)
plt.tight_layout()
out = os.path.join(OUT_DIR, 'coverage_map_comparison.png')
plt.savefig(out, dpi=150); plt.show()
print(f'Saved -> {out}')


## CELL 10a · RSSI / Path Loss / SINR Comparison

In [ ]:
# ====================================================================
# CELL 10a — RSSI / PATH LOSS / SINR COMPARISON (WITH vs WITHOUT SCATTER)
# ====================================================================
import matplotlib.pyplot as plt, numpy as np, os

print('=' * 70)
print('CELL 10a — PROPAGATION COMPARISON')
print('=' * 70)

# ── Noise floor and SINR ─────────────────────────────────────────────────────
NF_DB        = 5.0       # noise figure (dB)
kT_dBm_Hz    = -174.0   # thermal noise PSD (dBm/Hz)
noise_dbm    = kT_dBm_Hz + 10 * np.log10(BANDWIDTH_HZ) + NF_DB
print(f'Noise floor: {noise_dbm:.1f} dBm  (BW={BANDWIDTH_HZ/1e6:.0f} MHz, NF={NF_DB} dB)')

sinr_scatter    = rssi_scatter    - noise_dbm
sinr_no_scatter = rssi_no_scatter - noise_dbm

ext = [gx_min, gx_max, gy_min, gy_max]
tx  = list(scene.transmitters.values())[0]
tx_x = _safe(tx.position[0]); tx_y = _safe(tx.position[1])
sample_step = max(1, len(receivers) // 200)
rx_plot_x = [_safe(r.position[0]) for r in receivers[::sample_step]]
rx_plot_y = [_safe(r.position[1]) for r in receivers[::sample_step]]

# ── 3x2 map grid ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 2, figsize=(14, 15))
for ax, data, title, cmap, vmin, vmax, unit in [
    (axes[0, 0], rssi_scatter,        'RSSI – With scatter',        'jet',      -120, -40, 'dBm'),
    (axes[0, 1], rssi_no_scatter,     'RSSI – No scatter',          'jet',      -120, -40, 'dBm'),
    (axes[1, 0], path_loss_scatter,   'Path Loss – With scatter',   'plasma_r',   60, 140, 'dB'),
    (axes[1, 1], path_loss_no_scatter,'Path Loss – No scatter',     'plasma_r',   60, 140, 'dB'),
    (axes[2, 0], sinr_scatter,        'SINR – With scatter',        'RdYlGn',    -10,  30, 'dB'),
    (axes[2, 1], sinr_no_scatter,     'SINR – No scatter',          'RdYlGn',    -10,  30, 'dB'),
]:
    im = ax.imshow(data, origin='lower', extent=ext,
                   cmap=cmap, aspect='auto', vmin=vmin, vmax=vmax)
    ax.scatter(tx_x, tx_y, marker='*', s=200, c='gold', edgecolors='black')
    ax.scatter(rx_plot_x, rx_plot_y, s=3, c='cyan', alpha=0.5, label='RX')
    ax.set_title(title); ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)')
    ax.legend(loc='upper right', fontsize=7)
    plt.colorbar(im, ax=ax, label=unit)

plt.suptitle('Propagation Comparison', fontsize=14)
plt.tight_layout()
out = os.path.join(OUT_DIR, 'propagation_comparison.png')
plt.savefig(out, dpi=150); plt.show()
print(f'Saved -> {out}')

# ── Difference map: RSSI(scatter) – RSSI(no scatter) ─────────────────────────
rssi_diff = rssi_scatter - rssi_no_scatter
fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(rssi_diff, origin='lower', extent=ext,
               cmap='RdBu_r', vmin=-5, vmax=5)
ax.scatter(tx_x, tx_y, marker='*', s=200, c='gold', edgecolors='black', label='TX')
ax.scatter(rx_plot_x, rx_plot_y, s=5, c='cyan', alpha=0.5, label='RX')
ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)')
ax.set_title('Impact of scattering: ΔRSSI = With – Without (dB)')
plt.colorbar(im, label='dB'); ax.legend()
plt.tight_layout()
out2 = os.path.join(OUT_DIR, 'scattering_impact_map.png')
plt.savefig(out2, dpi=150); plt.show()
print(f'Saved -> {out2}')

# ── Per-receiver scatter comparison ──────────────────────────────────────────
rx_rssi_scatter    = df_scatter['rssi_dbm'].values
rx_rssi_no_scatter = df_no_scatter['rssi_dbm'].values
rx_pl_scatter      = df_scatter['path_loss_db'].values
rx_pl_no_scatter   = df_no_scatter['path_loss_db'].values

diff_rx = rx_rssi_scatter - rx_rssi_no_scatter
fig, axes2 = plt.subplots(1, 2, figsize=(12, 5))
axes2[0].scatter(rx_rssi_no_scatter, rx_rssi_scatter, alpha=0.5, s=15)
axes2[0].plot([-140, -20], [-140, -20], 'k--', label='1:1 line')
axes2[0].set_xlabel('RSSI no scatter (dBm)'); axes2[0].set_ylabel('RSSI with scatter (dBm)')
axes2[0].set_title('RSSI at receiver positions'); axes2[0].legend(); axes2[0].grid(True)
axes2[1].hist(diff_rx, bins=30, edgecolor='black', alpha=0.7)
axes2[1].axvline(0, color='r', linestyle='--', label='Zero diff')
axes2[1].set_xlabel('ΔRSSI = with − without (dB)'); axes2[1].set_ylabel('Count')
axes2[1].set_title(f'Scatter impact: mean={np.mean(diff_rx):.2f} dB  std={np.std(diff_rx):.2f} dB')
axes2[1].legend()
plt.tight_layout()
out3 = os.path.join(OUT_DIR, 'receiver_scatter_comparison.png')
plt.savefig(out3, dpi=150); plt.show()
print(f'Saved -> {out3}')

for label, r in [('With scatter', rx_rssi_scatter), ('Without scatter', rx_rssi_no_scatter)]:
    print(f'{label:20s}: mean={r.mean():.1f}  std={r.std():.1f}  '
          f'min={r.min():.1f}  max={r.max():.1f} dBm')


## CELL 9b · Path Computation (Sionna 0.19)

In [ ]:
# ====================================================================
# CELL 9b — PATH SOLVER WITH PER-RAY EXTRACTION  [Sionna 0.19]
# ====================================================================
# Ported from reference notebook CELL 11 (Sionna 2.0 PathSolver).
# Sionna 0.19 API: scene.compute_paths() replaces PathSolver().
#   - reflection   = specular_reflection (2.0)
#   - scattering   = diffuse_reflection  (2.0)
#   - refraction / edge_diffraction not available in 0.19
# Batching: process receivers in small groups to stay inside GPU budget.
# ====================================================================
import gc, time, os, numpy as np, pandas as pd, drjit as dr
from datetime import datetime

print('=' * 70)
print('CELL 9b — PATH SOLVER  [Sionna 0.19]')
print('=' * 70)

# ── Configuration ─────────────────────────────────────────────────────────────
SAVE_PER_RAY    = True
MAX_RAYS_PER_RX = 300
BATCH_SIZE      = 10      # receivers per compute_paths() call (tune for VRAM)
C               = 3e8     # speed of light m/s

PS_CONFIG = dict(
    max_depth   = MAX_DEPTH,
    num_samples = NUM_SAMPLES_PS,   # reduce in CELL 0c if still OOM
    los         = True,
    reflection  = True,             # specular_reflection in Sionna 2.0
    scattering  = True,             # diffuse_reflection  in Sionna 2.0
    diffraction = False,            # True = knife-edge, much slower
)

tx = list(scene.transmitters.values())[0]
tx_pwr_dbm = _safe(tx.power_dbm) if hasattr(tx, 'power_dbm') else EIRP_DBM
tx_pos     = np.array([_safe(tx.position[0]),
                       _safe(tx.position[1]),
                       _safe(tx.position[2])])
rx_gain_total = RX_GAIN_DBI + LNA_GAIN_DB

print(f'  TX EIRP         : {tx_pwr_dbm:.1f} dBm')
print(f'  TX position     : ({tx_pos[0]:.1f}, {tx_pos[1]:.1f}, {tx_pos[2]:.1f}) m')
print(f'  RX gain total   : {rx_gain_total:.1f} dB')
print(f'  Batch size      : {BATCH_SIZE} receivers')
print(f'  num_samples     : {NUM_SAMPLES_PS:,}')
print()
for k, v in PS_CONFIG.items():
    print(f'  {k:15s}: {v}')

# ── CIR helpers ───────────────────────────────────────────────────────────────
def extract_amplitudes(paths):
    """Return complex array (num_rx, max_num_paths)."""
    a = paths.a
    if isinstance(a, tuple):
        a_np = a[0].numpy() + 1j * a[1].numpy()
    else:
        a_np = np.array(a)
    # Typical shape: [num_rx, rx_ant, num_tx, tx_ant, num_paths] or [..., num_time]
    a_np = np.squeeze(a_np)
    # Collapse antenna and time dims, keep (num_rx, num_paths)
    while a_np.ndim > 2:
        a_np = a_np[..., 0]       # take first antenna / time slice
    if a_np.ndim == 1:
        a_np = a_np[np.newaxis, :]
    return a_np   # (num_rx, num_paths)

def extract_tau(paths, num_rx, num_paths):
    tau = getattr(paths, 'tau', None)
    if tau is None:
        return np.full((num_rx, num_paths), np.nan, np.float32)
    try:
        tau_np = tau.numpy() if hasattr(tau, 'numpy') else np.array(tau)
        tau_np = np.squeeze(tau_np)
        while tau_np.ndim > 2:
            tau_np = tau_np[..., 0]
        if tau_np.ndim == 1:
            tau_np = tau_np[np.newaxis, :]
        return tau_np
    except Exception:
        return np.full((num_rx, num_paths), np.nan, np.float32)

def summary_metrics(a_row):
    """Return (best_pl, incoherent_pl, coherent_pl, n_valid_paths)."""
    pwr   = np.abs(a_row) ** 2
    valid = pwr > 1e-30
    if not np.any(valid):
        return np.nan, np.nan, np.nan, 0
    pv = pwr[valid]
    av = a_row[valid]
    best_pl        = -10 * np.log10(np.max(pv))
    incoherent_pl  = -10 * np.log10(np.sum(pv))
    coherent_pwr   = np.abs(np.sum(av)) ** 2
    coherent_pl    = -10 * np.log10(coherent_pwr) if coherent_pwr > 1e-30 else np.nan
    return best_pl, incoherent_pl, coherent_pl, int(np.sum(valid))

def ray_type_heuristic(path_len, los_dist, pwr, max_pwr):
    if los_dist > 0 and abs(path_len - los_dist) / los_dist < 0.01:
        return 'LOS'
    ratio = pwr / max_pwr if max_pwr > 0 else 0
    excess = path_len - los_dist
    if excess < 50  and ratio > 0.01:  return 'REFLECTION'
    if excess >= 50 and ratio > 0.001: return 'MULTI_REFLECTION'
    if ratio < 0.01:                   return 'DIFFRACTION'
    if ratio < 0.001:                  return 'SCATTERING'
    return 'UNKNOWN'

# ── Batch solving loop ────────────────────────────────────────────────────────
total          = len(receivers)
ts             = datetime.now().strftime('%Y%m%d_%H%M%S')
summary_csv    = os.path.join(OUT_DIR, f'path_solver_summary_{ts}.csv')
per_ray_csv    = os.path.join(OUT_DIR, f'path_solver_per_ray_{ts}.csv') if SAVE_PER_RAY else None

# Snapshot current scene receivers so we can restore later
_all_rx = list(receivers)   # receivers list from CELL 7

summary_rows = []
per_ray_rows = []
errors       = 0
t0           = time.time()

print(f'\nProcessing {total} receivers in batches of {BATCH_SIZE} ...')

for b_start in range(0, total, BATCH_SIZE):
    batch = _all_rx[b_start : b_start + BATCH_SIZE]

    # Swap receivers in scene
    for name in list(scene.receivers.keys()):
        scene.remove(name)
    for rx in batch:
        scene.add(rx)

    try:
        paths = scene.compute_paths(**PS_CONFIG)
        a_all = extract_amplitudes(paths)             # (n_rx_batch, n_paths)
        n_b, n_p = a_all.shape
        tau_all = extract_tau(paths, n_b, n_p)

        for i, rx in enumerate(batch):
            rx_pos   = np.array([_safe(rx.position[0]),
                                 _safe(rx.position[1]),
                                 _safe(rx.position[2])])
            los_dist = float(np.linalg.norm(rx_pos - tx_pos))
            idx = i if i < n_b else n_b - 1

            best_pl, incoh_pl, coh_pl, n_valid = summary_metrics(a_all[idx])

            summary_rows.append({
                'receiver'              : rx.name,
                'x_m'                  : _safe(rx.position[0]),
                'y_m'                  : _safe(rx.position[1]),
                'z_m'                  : _safe(rx.position[2]),
                'dist_from_tx_m'       : los_dist,
                'num_paths'            : n_valid,
                'path_loss_best_db'    : best_pl,
                'path_loss_incoherent_db': incoh_pl,
                'path_loss_coherent_db': coh_pl,
                'rssi_best_dbm'        : (tx_pwr_dbm - best_pl   + rx_gain_total) if not np.isnan(best_pl)  else np.nan,
                'rssi_incoherent_dbm'  : (tx_pwr_dbm - incoh_pl  + rx_gain_total) if not np.isnan(incoh_pl) else np.nan,
                'rssi_coherent_dbm'    : (tx_pwr_dbm - coh_pl    + rx_gain_total) if not np.isnan(coh_pl)   else np.nan,
            })

            if SAVE_PER_RAY and n_valid > 0:
                a_row  = a_all[idx]
                pwr_row = np.abs(a_row) ** 2
                order  = np.argsort(pwr_row)[::-1]
                max_pwr = pwr_row[order[0]]
                strong_phase = np.angle(a_row[order[0]], deg=True)

                for rank, ray_i in enumerate(order[:MAX_RAYS_PER_RX]):
                    ac      = a_row[ray_i]
                    pwr_ray = float(pwr_row[ray_i])
                    if pwr_ray <= 1e-30:
                        break
                    phase   = float(np.angle(ac, deg=True))
                    ph_diff = (phase - strong_phase + 180) % 360 - 180
                    delay   = float(tau_all[idx, ray_i]) if not np.isnan(tau_all[idx, ray_i]) else np.nan
                    plen    = delay * C if not np.isnan(delay) else np.nan
                    rtype   = ray_type_heuristic(plen, los_dist, pwr_ray, max_pwr) \
                              if not np.isnan(plen) else 'UNKNOWN'
                    per_ray_rows.append({
                        'receiver'          : rx.name,
                        'rank'              : rank,
                        'ray_type'          : rtype,
                        'power_linear'      : pwr_ray,
                        'path_loss_db'      : -10 * np.log10(pwr_ray),
                        'amplitude_real'    : float(ac.real),
                        'amplitude_imag'    : float(ac.imag),
                        'phase_deg'         : phase,
                        'phase_diff_deg'    : ph_diff,
                        'constructive'      : 'STRONGEST' if rank == 0 else
                                             ('CONSTRUCTIVE' if abs(ph_diff) < 90 else 'DESTRUCTIVE'),
                        'delay_s'           : delay,
                        'path_length_m'     : plen,
                    })

        del paths, a_all, tau_all
    except Exception as _e:
        print(f'  [WARN] Batch {b_start}: {_e}')
        for rx in batch:
            los_dist = float(np.linalg.norm(
                np.array([_safe(rx.position[0]),_safe(rx.position[1]),_safe(rx.position[2])]) - tx_pos))
            summary_rows.append({'receiver': rx.name,
                'x_m': _safe(rx.position[0]), 'y_m': _safe(rx.position[1]),
                'z_m': _safe(rx.position[2]), 'dist_from_tx_m': los_dist,
                'num_paths': 0, 'path_loss_best_db': np.nan,
                'path_loss_incoherent_db': np.nan, 'path_loss_coherent_db': np.nan,
                'rssi_best_dbm': np.nan, 'rssi_incoherent_dbm': np.nan, 'rssi_coherent_dbm': np.nan})
        errors += len(batch)

    gc.collect()
    done = min(b_start + BATCH_SIZE, total)
    if done % max(BATCH_SIZE, total // 10) < BATCH_SIZE or done == total:
        elapsed = time.time() - t0
        eta     = (total - done) / max(done / elapsed, 1e-9)
        print(f'  [{done}/{total}]  {elapsed:.0f}s elapsed  ETA {eta/60:.1f} min', flush=True)

# Restore all receivers
for name in list(scene.receivers.keys()):
    scene.remove(name)
for rx in _all_rx:
    scene.add(rx)

# ── Save ──────────────────────────────────────────────────────────────────────
df_ps = pd.DataFrame(summary_rows)
df_ps.to_csv(summary_csv, index=False)
print(f'\n  Summary  -> {summary_csv}')

if SAVE_PER_RAY and per_ray_rows:
    df_ray = pd.DataFrame(per_ray_rows)
    df_ray.to_csv(per_ray_csv, index=False)
    print(f'  Per-ray  -> {per_ray_csv}  ({len(df_ray):,} rays)')

elapsed = time.time() - t0
print(f'  Total time: {elapsed:.1f}s  |  Errors: {errors}')

valid = df_ps.dropna(subset=['path_loss_incoherent_db'])
if len(valid):
    for col, lbl in [('path_loss_incoherent_db','PL incoherent'),
                     ('path_loss_coherent_db',  'PL coherent'),
                     ('rssi_incoherent_dbm',    'RSSI incoher.')]:
        if col in valid:
            v = valid[col].dropna()
            if len(v):
                print(f'  {lbl:20s}: mean={v.mean():.1f}  std={v.std():.1f}  '
                      f'min={v.min():.1f}  max={v.max():.1f}')


## CELL 11 · Path Loss vs Distance

In [ ]:
tx_pos2d = np.array([_safe(tx.position[0]), _safe(tx.position[1])])
dist_m   = np.linalg.norm(df_out[['x_m','y_m']].values - tx_pos2d, axis=1)
df_out['dist_m'] = dist_m
df_out['path_loss'] = -df_out['pg_db']

_lam  = C / FREQUENCY_HZ
d_ref = np.linspace(max(dist_m.min(), 10), dist_m.max(), 300)
fspl  = 20*np.log10(4*np.pi*d_ref / _lam)

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(dist_m, df_out['path_loss'], s=5, alpha=0.4, c='steelblue', label='Simulated')
ax.plot(d_ref, fspl, 'r--', lw=2, label='Free-space PL')
ax.set_xlabel('Distance TX→RX (m)'); ax.set_ylabel('Path Loss (dB)')
ax.set_title(f'{CITY_NAME} – Path Loss @ {FREQUENCY_HZ/1e9:.2f} GHz')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'path_loss_vs_distance.png'), dpi=150)
plt.show()
print('All results saved to:', OUT_DIR)